In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

OUTPUT_DIR = Path(
    r"C:\Users\RAAJARAPU SUSWIN\Downloads\case-a-thon_dset\outputs"
)

print("Notebook 04 started")

Notebook 04 started


In [2]:
features = pd.read_csv(
    OUTPUT_DIR / "model_features.csv"
)

features["consult_date"] = pd.to_datetime(
    features["consult_date"],
    errors="coerce"
)

print("Dataset:", features.shape)
print(features["cohort"].value_counts())

Dataset: (5516, 27)
cohort
DEVELOPMENT    4132
EVALUATION     1384
Name: count, dtype: int64


In [3]:
exclude_columns = [
    "episode_id",
    "teleconsult_id",
    "patient_id",
    "consult_date",
    "cohort",
    "lost_to_followup_label"
]

feature_columns = [
    column
    for column in features.columns
    if column not in exclude_columns
]

X = features[feature_columns].copy()
y = features["lost_to_followup_label"].copy()

development_mask = (
    features["cohort"] == "DEVELOPMENT"
)

X_dev = X.loc[development_mask].copy()
y_dev = y.loc[development_mask].astype(int)

dates_dev = features.loc[
    development_mask,
    "consult_date"
].copy()

print("Development:", X_dev.shape)
print("Target distribution:")
print(y_dev.value_counts())

Development: (4132, 21)
Target distribution:
lost_to_followup_label
0    2144
1    1988
Name: count, dtype: int64


In [4]:
# Sort development data chronologically

order = dates_dev.sort_values().index

X_dev = X_dev.loc[order]
y_dev = y_dev.loc[order]
dates_dev = dates_dev.loc[order]

split_position = int(
    len(X_dev) * 0.80
)

X_train = X_dev.iloc[:split_position].copy()
X_valid = X_dev.iloc[split_position:].copy()

y_train = y_dev.iloc[:split_position].copy()
y_valid = y_dev.iloc[split_position:].copy()

print("Training rows:", len(X_train))
print("Validation rows:", len(X_valid))

print(
    "\nTraining period:",
    dates_dev.iloc[0],
    "to",
    dates_dev.iloc[split_position - 1]
)

print(
    "Validation period:",
    dates_dev.iloc[split_position],
    "to",
    dates_dev.iloc[-1]
)

print("\nTraining target:")
print(y_train.value_counts(normalize=True))

print("\nValidation target:")
print(y_valid.value_counts(normalize=True))

Training rows: 3305
Validation rows: 827

Training period: 2026-01-01 00:00:00 to 2026-05-31 00:00:00
Validation period: 2026-06-01 00:00:00 to 2026-06-30 00:00:00

Training target:
lost_to_followup_label
0    0.514977
1    0.485023
Name: proportion, dtype: float64

Validation target:
lost_to_followup_label
0    0.534462
1    0.465538
Name: proportion, dtype: float64


In [5]:
models = {

    "Logistic Regression": Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        ),
        (
            "model",
            LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=42
            )
        )
    ]),

    "Random Forest": Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "model",
            RandomForestClassifier(
                n_estimators=300,
                max_depth=10,
                min_samples_leaf=5,
                class_weight="balanced",
                random_state=42,
                n_jobs=-1
            )
        )
    ]),

    "Extra Trees": Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "model",
            ExtraTreesClassifier(
                n_estimators=300,
                max_depth=12,
                min_samples_leaf=4,
                class_weight="balanced",
                random_state=42,
                n_jobs=-1
            )
        )
    ])
}

print("Models:")
for name in models:
    print("-", name)

Models:
- Logistic Regression
- Random Forest
- Extra Trees


In [6]:
results = []
predictions = {}

for name, model in models.items():

    print(f"\nTraining: {name}")

    model.fit(
        X_train,
        y_train
    )

    probability = model.predict_proba(
        X_valid
    )[:, 1]

    prediction = (
        probability >= 0.50
    ).astype(int)

    predictions[name] = probability

    results.append({
        "model": name,
        "ROC_AUC": roc_auc_score(
            y_valid,
            probability
        ),
        "PR_AUC": average_precision_score(
            y_valid,
            probability
        ),
        "Accuracy": accuracy_score(
            y_valid,
            prediction
        ),
        "Precision": precision_score(
            y_valid,
            prediction,
            zero_division=0
        ),
        "Recall": recall_score(
            y_valid,
            prediction,
            zero_division=0
        ),
        "F1": f1_score(
            y_valid,
            prediction,
            zero_division=0
        )
    })

results_df = pd.DataFrame(results)

display(
    results_df.sort_values(
        "PR_AUC",
        ascending=False
    )
)


Training: Logistic Regression

Training: Random Forest

Training: Extra Trees


,model,ROC_AUC,PR_AUC,Accuracy,Precision,Recall,F1
0,Logistic Regression,0.702674,0.671655,0.648126,0.615764,0.649351,0.632111
2,Extra Trees,0.697502,0.653645,0.644498,0.612346,0.644156,0.627848
1,Random Forest,0.691603,0.646889,0.634825,0.605598,0.618182,0.611825


In [7]:
results_sorted = results_df.sort_values(
    "PR_AUC",
    ascending=False
).reset_index(drop=True)

best_model_name = results_sorted.loc[
    0,
    "model"
]

best_model = models[
    best_model_name
]

print("Selected model based on validation PR-AUC:")
print(best_model_name)

display(results_sorted)

Selected model based on validation PR-AUC:
Logistic Regression


,model,ROC_AUC,PR_AUC,Accuracy,Precision,Recall,F1
0,Logistic Regression,0.702674,0.671655,0.648126,0.615764,0.649351,0.632111
1,Extra Trees,0.697502,0.653645,0.644498,0.612346,0.644156,0.627848
2,Random Forest,0.691603,0.646889,0.634825,0.605598,0.618182,0.611825


In [8]:
best_probability = predictions[
    best_model_name
]

best_prediction = (
    best_probability >= 0.50
).astype(int)

cm = confusion_matrix(
    y_valid,
    best_prediction
)

print("Confusion Matrix")
print(cm)

print("\nClassification Report")
print(
    classification_report(
        y_valid,
        best_prediction,
        target_names=[
            "Completed",
            "LTFU"
        ],
        zero_division=0
    )
)

Confusion Matrix
[[286 156]
 [135 250]]

Classification Report
              precision    recall  f1-score   support

   Completed       0.68      0.65      0.66       442
        LTFU       0.62      0.65      0.63       385

    accuracy                           0.65       827
   macro avg       0.65      0.65      0.65       827
weighted avg       0.65      0.65      0.65       827



In [9]:
validation_results = features.loc[
    y_valid.index,
    [
        "episode_id",
        "patient_id",
        "consult_date"
    ]
].copy()

validation_results["actual_ltfu"] = y_valid.values

validation_results["risk_probability"] = (
    best_probability
)

validation_results["risk_tier"] = pd.cut(
    validation_results["risk_probability"],
    bins=[
        -0.01,
        0.30,
        0.60,
        0.80,
        1.01
    ],
    labels=[
        "LOW",
        "MEDIUM",
        "HIGH",
        "VERY HIGH"
    ]
)

display(
    validation_results.head(20)
)

,episode_id,patient_id,consult_date,actual_ltfu,risk_probability,risk_tier
2391,E0002392,P002170,2026-06-01,0,0.334868,MEDIUM
3742,E0003743,P003403,2026-06-01,0,0.399666,MEDIUM
1658,E0001659,P001508,2026-06-01,0,0.646030,HIGH
1858,E0001859,P001686,2026-06-01,1,0.651689,HIGH
168,E0000169,P000146,2026-06-01,1,0.435273,MEDIUM
5148,E0005149,P004670,2026-06-01,1,0.621642,HIGH
1547,E0001548,P001412,2026-06-01,1,0.830375,VERY HIGH
2591,E0002592,P002343,2026-06-01,0,0.469074,MEDIUM
5288,E0005289,P004800,2026-06-01,0,0.395924,MEDIUM
389,E0000390,P000343,2026-06-01,1,0.129185,LOW


In [11]:
risk_summary = (
    validation_results
    .groupby(
        "risk_tier",
        observed=False
    )
    .agg(
        episodes=("episode_id", "count"),
        actual_ltfu_rate=("actual_ltfu", "mean")
    )
    .reset_index()
)

risk_summary["actual_ltfu_rate"] *= 100

risk_summary["actual_ltfu_rate"] = (
    risk_summary["actual_ltfu_rate"]
    .round(2)
)

display(risk_summary)

,risk_tier,episodes,actual_ltfu_rate
0,LOW,146,23.29
1,MEDIUM,403,41.44
2,HIGH,244,63.93
3,VERY HIGH,34,82.35


In [12]:
results_df.to_csv(
    OUTPUT_DIR / "model_comparison.csv",
    index=False
)

risk_summary.to_csv(
    OUTPUT_DIR / "risk_tier_validation.csv",
    index=False
)

print("Model comparison saved.")

Model comparison saved.


In [13]:
best_model.fit(
    X_dev,
    y_dev
)

print(
    "Final model trained:",
    best_model_name
)

Final model trained: Logistic Regression


In [14]:
evaluation_mask = (
    features["cohort"] == "EVALUATION"
)

X_eval = features.loc[
    evaluation_mask,
    feature_columns
].copy()

evaluation_predictions = best_model.predict_proba(
    X_eval
)[:, 1]

evaluation_output = features.loc[
    evaluation_mask,
    [
        "episode_id",
        "teleconsult_id",
        "patient_id",
        "consult_date"
    ]
].copy()

evaluation_output["risk_probability"] = (
    evaluation_predictions
)

evaluation_output["predicted_lost_to_followup"] = (
    evaluation_output["risk_probability"] >= 0.50
).astype(int)

evaluation_output["priority_tier"] = pd.cut(
    evaluation_output["risk_probability"],
    bins=[
        -0.01,
        0.30,
        0.60,
        0.80,
        1.01
    ],
    labels=[
        "LOW",
        "MEDIUM",
        "HIGH",
        "VERY HIGH"
    ]
)

display(
    evaluation_output.head(20)
)

,episode_id,teleconsult_id,patient_id,consult_date,risk_probability,predicted_lost_to_followup,priority_tier
1,E0000002,TC0000002,P000004,2026-07-22,0.401178,0,MEDIUM
2,E0000003,TC0000003,P000004,2026-08-11,0.357278,0,MEDIUM
8,E0000009,TC0000009,P000008,2026-07-19,0.563782,1,MEDIUM
13,E0000014,TC0000014,P000010,2026-07-12,0.716091,1,HIGH
17,E0000018,TC0000018,P000014,2026-07-01,0.687923,1,HIGH
18,E0000019,TC0000019,P000014,2026-08-07,0.312529,0,MEDIUM
20,E0000021,TC0000021,P000017,2026-08-12,0.355052,0,MEDIUM
21,E0000022,TC0000022,P000018,2026-07-23,0.410201,0,MEDIUM
22,E0000023,TC0000023,P000019,2026-07-22,0.354187,0,MEDIUM
23,E0000024,TC0000024,P000019,2026-07-29,0.616694,1,HIGH


In [15]:
evaluation_output.to_csv(
    OUTPUT_DIR / "evaluation_predictions.csv",
    index=False
)

print(
    "Evaluation predictions saved:",
    len(evaluation_output)
)

Evaluation predictions saved: 1384


In [16]:
print("=" * 60)
print("NOTEBOOK 04 COMPLETE")
print("=" * 60)

print("\nMODEL COMPARISON")
display(
    results_sorted
)

print(
    "\nSelected model:",
    best_model_name
)

print(
    "\nValidation PR-AUC:",
    round(
        results_sorted.loc[
            0,
            "PR_AUC"
        ],
        4
    )
)

print(
    "\nEvaluation predictions:",
    len(evaluation_output)
)

print("\nSaved:")
print("model_comparison.csv")
print("risk_tier_validation.csv")
print("evaluation_predictions.csv")

NOTEBOOK 04 COMPLETE

MODEL COMPARISON


,model,ROC_AUC,PR_AUC,Accuracy,Precision,Recall,F1
0,Logistic Regression,0.702674,0.671655,0.648126,0.615764,0.649351,0.632111
1,Extra Trees,0.697502,0.653645,0.644498,0.612346,0.644156,0.627848
2,Random Forest,0.691603,0.646889,0.634825,0.605598,0.618182,0.611825



Selected model: Logistic Regression

Validation PR-AUC: 0.6717

Evaluation predictions: 1384

Saved:
model_comparison.csv
risk_tier_validation.csv
evaluation_predictions.csv
